
# DSS-LVR IAF regression diagnostics

This notebook has two separate purposes:

1. **Current controlled training:** use an independent RNG for `q0.loc` jitter, isolate diagnostic sampling from the training RNG, and record how the transported posterior changes with flow depth.
2. **Historical IAF reproduction:** reproduce the original `iaf_ordering_depth_test` conditions as closely as possible, including its old RNG behavior, so changes made after that benchmark can be detected.

The historical section is intentionally isolated. Do not use its legacy RNG behavior for new experiments.


In [1]:
from pathlib import Path
import inspect
import math
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from IPython.display import display


def find_project_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "Python" / "model2.py").exists() and (root / "Python" / "bnn_metric.py").exists():
            return root
    raise FileNotFoundError("Run this notebook inside the NFlow repository.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import Python.model2 as md
import Python.bnn_metric as metric

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
HAS_SLAB_INIT = "slab_init" in inspect.signature(md.GroupedBNNVI).parameters

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("project root :", ROOT)
print("device       :", DEVICE)
print("slab_init API:", HAS_SLAB_INIT)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
project root : D:\positron\NFlow
device       : cpu
slab_init API: True


## Current simulator

In [2]:
def simfun(
    n=600,
    p=100,
    n_active=10,
    n_true_units=10,
    activation="relu",          # "none", "relu", or "trig"
    structure="mixed",          # "axis" or "mixed"
    features_per_unit=3,        # only used for mixed
    n_interactions=0,           # 0, 1, 2, 3
    n_quadratic=0,              # 0, 1, 2, 3
    extra_scale=0.5,
    sigma2=1.0,
    target_signal_sd=1.5,
    x_low=-2.5,
    x_high=2.5,
    seed=123,
    device=None,
    dtype=torch.float32,
):
    device = torch.device("cpu") if device is None else torch.device(device)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    n, p, n_active, K = int(n), int(p), int(n_active), int(n_true_units)
    n_interactions, n_quadratic = int(n_interactions), int(n_quadratic)

    if activation not in {"none", "relu", "trig"}:
        raise ValueError("activation must be 'none', 'relu', or 'trig'.")
    if structure not in {"axis", "mixed"}:
        raise ValueError("structure must be 'axis' or 'mixed'.")
    if not 0 <= n_interactions <= 3 or not 0 <= n_quadratic <= 3:
        raise ValueError("n_interactions and n_quadratic must be between 0 and 3.")

    X = float(x_low) + (float(x_high) - float(x_low)) * torch.rand(
        n, p, generator=gen, device=device, dtype=dtype
    )

    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))
    active_t = torch.as_tensor(active_idx, device=device, dtype=torch.long)
    Xa = X.index_select(1, active_t)

    W = np.zeros((K, n_active), dtype=np.float32)

    if structure == "axis":
        if K < n_active:
            raise ValueError("axis requires n_true_units >= n_active.")
        supports = [{unit % n_active} for unit in range(K)]
    else:
        m = min(int(features_per_unit), n_active)
        if K * m < n_active:
            raise ValueError("mixed requires n_true_units * features_per_unit >= n_active.")

        supports = [set() for _ in range(K)]
        for pos, feature in enumerate(rng.permutation(n_active)):
            supports[pos % K].add(int(feature))
        for unit in range(K):
            while len(supports[unit]) < m:
                supports[unit].add(int(rng.integers(n_active)))

    for unit, support in enumerate(supports):
        idx = np.asarray(sorted(support), dtype=int)
        signs = rng.choice([-1.0, 1.0], size=len(idx))
        magnitude = rng.uniform(0.8, 1.2)
        W[unit, idx] = magnitude * signs / np.sqrt(len(idx))

    if activation in {"relu", "trig"}:
        bias = rng.uniform(-0.8, 0.8, size=K).astype(np.float32)
    else:
        bias = np.zeros(K, dtype=np.float32)

    amplitude = rng.uniform(0.8, 1.2, size=K).astype(np.float32)

    Wt = torch.as_tensor(W, device=device, dtype=dtype)
    bt = torch.as_tensor(bias, device=device, dtype=dtype)
    at = torch.as_tensor(amplitude, device=device, dtype=dtype)

    pre = Xa @ Wt.T - bt

    trig_functions = []
    if activation == "relu":
        hidden = F.relu(pre)
    elif activation == "trig":
        hidden = torch.empty_like(pre)
        trig_names = ("sin", "cos", "sin2", "cos2")
        for unit in range(K):
            mode = unit % 4
            z = pre[:, unit]
            if mode == 0:
                hidden[:, unit] = torch.sin(z)
            elif mode == 1:
                hidden[:, unit] = torch.cos(z)
            elif mode == 2:
                hidden[:, unit] = torch.sin(z).square()
            else:
                hidden[:, unit] = torch.cos(z).square()
            trig_functions.append(trig_names[mode])
    else:
        hidden = pre

    signal = hidden @ at

    interaction_pairs = []
    if n_interactions > 0:
        candidates = [(j, k) for j in range(n_active) for k in range(j + 1, n_active)]
        rng.shuffle(candidates)
        interaction_pairs = candidates[:n_interactions]

        for j, k in interaction_pairs:
            term = Xa[:, j] * Xa[:, k]
            term = (term - term.mean()) / term.std(unbiased=False).clamp_min(1e-8)
            signal = signal + float(extra_scale) * term

    quadratic_features = []
    if n_quadratic > 0:
        quadratic_features = rng.choice(
            n_active, size=min(n_quadratic, n_active), replace=False
        ).tolist()

        for j in quadratic_features:
            term = Xa[:, j].square()
            term = (term - term.mean()) / term.std(unbiased=False).clamp_min(1e-8)
            signal = signal + float(extra_scale) * term

    signal = signal - signal.mean()
    signal = signal * float(target_signal_sd) / signal.std(unbiased=False).clamp_min(1e-8)

    y = signal + math.sqrt(float(sigma2)) * torch.randn(
        n, generator=gen, device=device, dtype=dtype
    )

    feature_true = torch.zeros(p, device=device, dtype=dtype)
    feature_true[active_t] = 1.0

    info = {
        "sim": f"{structure}_{activation}",
        "seed": int(seed),
        "n": n,
        "p": p,
        "n_active": n_active,
        "active_idx": active_idx,
        "feature_true": feature_true.detach().cpu().numpy(),
        "n_true_units": K,
        "activation": activation,
        "structure": structure,
        "features_per_unit": 1 if structure == "axis" else int(features_per_unit),
        "teacher_supports": [
            active_idx[np.asarray(sorted(s), dtype=int)].tolist()
            for s in supports
        ],
        "trig_functions": trig_functions,
        "interaction_pairs": [
            (int(active_idx[j]), int(active_idx[k]))
            for j, k in interaction_pairs
        ],
        "quadratic_features": [
            int(active_idx[j])
            for j in quadratic_features
        ],
        "n_interactions": len(interaction_pairs),
        "n_quadratic": len(quadratic_features),
        "sigma2": float(sigma2),
        "signal_sd": float(signal.std(unbiased=False)),
    }

    return X, y, feature_true, signal, info


## Controlled trainer with flow-depth diagnostics

The production path below always uses an **independent jitter RNG**. Diagnostic draws also run inside `torch.random.fork_rng`, so changing `record_every` or `R_diag` cannot alter subsequent optimization draws.

Recorded checkpoints distinguish the base posterior (`q0_*_sd`) from the transported posterior (`post_*_sd`), and separately track all-on representation R² (`repr_r2`) versus the actually gated network (`gated_r2`).


In [3]:
def train_trace(
    X_train, y_train, X_eval, signal_eval, X_test, signal_test, *, truth,
    selection_mode="feature_group", hidden_dims=(20,), sigma2=1.0,
    init_sd=0.5, K_flow=4, flow_hidden_units=128, flow_hidden_layers=2,
    scale_clip=2.0, flow_seed=123, iaf_ordering_scheme="cyclic3",
    iaf_shuffle_within_role=True, gate_type="normalized_requ", gate_scale=1.0,
    epochs=1200, warmup_epochs=300, lr=3e-4, R_train=32, R_diag=128,
    R_final=500, record_every=100, init_loc_jitter=0.05, jitter_seed_offset=99173,
    grad_clip=5.0, support_threshold=0.5, seed=123,
    slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

    model_kwargs = dict(
        X=X_train, y=y_train, input_dim=X_train.shape[1], hidden_dims=tuple(hidden_dims), out_dim=1,
        selection_mode=selection_mode, family="gaussian", sigma2=float(sigma2), init_sd=float(init_sd),
        K_flow=int(K_flow), flow_type="iaf", flow_hidden_units=int(flow_hidden_units),
        flow_hidden_layers=int(flow_hidden_layers), scale_clip=float(scale_clip), flow_seed=int(flow_seed),
        iaf_ordering_scheme=iaf_ordering_scheme, iaf_shuffle_within_role=bool(iaf_shuffle_within_role),
        gate_type=gate_type, gate_scale=float(gate_scale),
    )
    if HAS_SLAB_INIT:
        model_kwargs.update(slab_init=slab_init, slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd)
    model = md.GroupedBNNVI(**model_kwargs).to(DEVICE)

    # Independent jitter RNG: changing K_flow no longer changes q0.loc through RNG consumption.
    if float(init_loc_jitter) > 0:
        g = torch.Generator(device=model.q0.loc.device)
        g.manual_seed(int(seed) + int(jitter_seed_offset))
        jitter = torch.randn(model.q0.loc.shape, generator=g, device=model.q0.loc.device, dtype=model.q0.loc.dtype)
        with torch.no_grad(): model.q0.loc.add_(float(init_loc_jitter) * jitter)

    init_q0_loc = model.q0.loc.detach().cpu().clone()
    init_q0_sd = torch.exp(model.q0.raw_log_scale.detach().clamp(-8.0, 2.0)).cpu().clone()
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = []

    def record(epoch, phase):
        model.eval()
        devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
        with torch.random.fork_rng(devices=devices):
            torch.manual_seed(int(seed) + 700000 + int(epoch))
            if DEVICE.type == "cuda": torch.cuda.manual_seed_all(int(seed) + 700000 + int(epoch))
            with torch.no_grad():
                xi, log_q = model.sample_posterior(int(R_diag))
                s, v, t = model.decoder.split_latent(xi)
                sem = model.decoder.group_semantics(xi)
                pred_repr = model.decoder(X_eval, xi, force_all_on=True)
                pred_gated = model.decoder(X_eval, xi, force_all_on=(phase in {"init", "repr"}))
                f_repr = metric.function_metrics(signal_eval, pred_repr)
                f_gated = metric.function_metrics(signal_eval, pred_gated)
                ll = model.log_likelihood(xi, force_all_on=(phase in {"init", "repr"})).mean()
                prior = model.log_prior(xi).mean()
                q0_sd = torch.exp(model.q0.raw_log_scale.detach().clamp(-8.0, 2.0))
                row = {
                    "epoch": int(epoch), "phase": phase,
                    "repr_r2": float(f_repr["r2"]), "gated_r2": float(f_gated["r2"]),
                    "repr_mse": float(f_repr["mse"]),
                    "pred_var": float(pred_repr.var(dim=0, unbiased=False).mean()),
                    "pred_mean_sd": float(pred_repr.mean(dim=0).std(unbiased=False)),
                    "post_U_sd": float(s.std(dim=0, unbiased=False).mean()),
                    "post_V_sd": float(v.std(dim=0, unbiased=False).mean()),
                    "post_tau_sd": float(t.std(dim=0, unbiased=False).mean()),
                    "q0_U_sd": float(q0_sd[:model.decoder.s_dim].mean()),
                    "q0_V_sd": float(q0_sd[model.decoder.s_dim:model.decoder.s_dim + model.decoder.u_dim].mean()),
                    "q0_tau_sd": float(q0_sd[model.decoder.s_dim + model.decoder.u_dim:].mean()),
                    "margin_mean": float(sem["margin"].mean()), "margin_sd": float(sem["margin"].std(unbiased=False)),
                    "active_rate": float(sem["active"].float().mean()),
                    "loglik": float(ll), "logprior": float(prior), "logq": float(log_q.mean()),
                    "kl_like": float((log_q.mean() - prior)),
                }
                if model.decoder.has_feature_gates:
                    fp = model.decoder.feature_semantics(xi)["active"].float().mean(0)
                    target = torch.as_tensor(np.asarray(truth["feature_true"]) > 0.5, device=fp.device)
                    row["feature_pip_mean"] = float(fp.mean())
                    row["active_feature_pip"] = float(fp[target].mean()) if target.any() else np.nan
                    row["inactive_feature_pip"] = float(fp[~target].mean()) if (~target).any() else np.nan
                if model.decoder.has_unit_gates:
                    up = model.decoder.unit_semantics(xi)["active"].float().mean(0)
                    row["unit_pip_mean"] = float(up.mean())
                    row["expected_units"] = float(up.sum())
        history.append(row)
        print(f"epoch={epoch:04d} {phase:6s} reprR2={row['repr_r2']:+.4f} gatedR2={row['gated_r2']:+.4f} "
              f"predVar={row['pred_var']:.3g} U/V/tauSD={row['post_U_sd']:.3g}/{row['post_V_sd']:.3g}/{row['post_tau_sd']:.3g} "
              f"active={row['active_rate']:.3f}")

    record(0, "init")
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, int(epochs) + 1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        warmup = epoch <= int(warmup_epochs)
        if warmup:
            xi, log_q = model.sample_posterior(int(R_train))
            elbo = model.log_likelihood(xi, force_all_on=True) + model.log_prior(xi) - log_q
        else:
            elbo = model.elbo_draws(int(R_train))["elbo"]
        (-elbo.mean()).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
        optimizer.step()
        if epoch == 1 or epoch % int(record_every) == 0 or epoch == int(warmup_epochs) or epoch == int(epochs):
            record(epoch, "repr" if warmup else "select")

    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_time = time.perf_counter() - started
    model.eval()
    with torch.no_grad():
        xi_final, _ = model.sample_posterior(int(R_final))
        pred_test = model.decoder(X_test, xi_final)
        fm = metric.function_metrics(signal_test, pred_test)

    result = {"mse": float(fm["mse"]), "r2": float(fm["r2"]), "train_time_sec": float(train_time)}
    feature_pip = unit_pip = None
    if model.decoder.has_feature_gates:
        feature_pip = model.decoder.feature_semantics(xi_final)["active"].float().mean(0).cpu().numpy()
        target = np.asarray(truth["feature_true"], dtype=float).reshape(-1) > 0.5
        selected = feature_pip > float(support_threshold); active, inactive = target, ~target
        ba = np.mean((1.0 - feature_pip[active]) ** 2); b0 = np.mean(feature_pip[inactive] ** 2) if inactive.any() else np.nan
        result.update(
            tpr=float(selected[active].mean()),
            fpr=float(selected[inactive].mean()) if inactive.any() else np.nan,
            accuracy=float(np.mean(selected == target)),
            auroc=float(roc_auc_score(target.astype(int), feature_pip))
                if np.unique(target).size == 2 else np.nan,
            brier_bal=float(0.5 * (ba + b0)) if inactive.any() else float(ba),
            expected_support=float(feature_pip.sum()),
            selected_support=int(selected.sum()),
            mean_active_pip=float(feature_pip[active].mean()),
            mean_inactive_pip=float(feature_pip[inactive].mean())
                if inactive.any() else np.nan,
        )
    if model.decoder.has_unit_gates:
        unit_pip = model.decoder.unit_semantics(xi_final)["active"].float().mean(0).cpu().numpy()
        result.update(expected_active_units=float(unit_pip.sum()),
                      selected_active_units=int((unit_pip > float(support_threshold)).sum()))

    # Hard MPM parameter density, biases excluded.
    p_input = int(X_train.shape[1])

    if feature_pip is not None:
        n_feature_mpm = int((feature_pip > float(support_threshold)).sum())
    else:
        n_feature_mpm = p_input

    if unit_pip is not None:
        unit_counts = []
        start = 0
        for h in hidden_dims:
            h = int(h)
            unit_counts.append(
                int((unit_pip[start:start + h] > float(support_threshold)).sum())
            )
            start += h
    else:
        unit_counts = [int(h) for h in hidden_dims]

    sparse_dims = [n_feature_mpm] + unit_counts + [1]
    dense_dims = [p_input] + [int(h) for h in hidden_dims] + [1]

    retained_weights = sum(
        sparse_dims[j] * sparse_dims[j + 1]
        for j in range(len(sparse_dims) - 1)
    )

    candidate_weights = sum(
        dense_dims[j] * dense_dims[j + 1]
        for j in range(len(dense_dims) - 1)
    )

    result["retained_weights"] = int(retained_weights)
    result["candidate_weights"] = int(candidate_weights)
    result["dparam"] = float(retained_weights / candidate_weights)

    if selection_mode == "feature_unit_induced_edge":
        result["network_density"] = float(metric.network_density(model.decoder, xi_final))
        result["path_density"] = float(metric.active_path_density(model.decoder, xi_final))

    return {"result": result, "history": pd.DataFrame(history), "model": model, "xi": xi_final.detach(),
            "feature_pip": feature_pip, "unit_pip": unit_pip,
            "init_q0_loc": init_q0_loc, "init_q0_sd": init_q0_sd}


In [5]:
def run_experiment(
    *, n=600, p=100, n_active=10, n_true_units=4, activation="relu", structure="mixed",
    features_per_unit=3, n_interactions=2, n_quadratic=1, extra_scale=0.5,
    target_signal_sd=1.5, sigma2=1.0, x_low=-2.5, x_high=2.5, data_seed=400,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=4, flow_hidden_units=128, flow_hidden_layers=2, scale_clip=2.0,
    iaf_ordering_scheme="cyclic3", iaf_shuffle_within_role=True,
    gate_type="normalized_requ", gate_scale=1.0, train_frac=0.8, diagnostic_frac=0.10,
    epochs=1200, warmup_epochs=300, lr=3e-4, R_train=32, R_diag=128, R_final=500,
    record_every=100, init_sd=0.5, init_loc_jitter=0.05, grad_clip=5.0,
    support_threshold=0.5, fit_seed=None, slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    if fit_seed is None: fit_seed = int(data_seed + 100_000 + p + 17 * len(tuple(hidden_dims)))
    X, y, feature_true, signal, info = simfun(
        n=n, p=p, n_active=n_active, n_true_units=n_true_units, activation=activation, structure=structure,
        features_per_unit=features_per_unit, n_interactions=n_interactions, n_quadratic=n_quadratic,
        extra_scale=extra_scale, sigma2=sigma2, target_signal_sd=target_signal_sd,
        x_low=x_low, x_high=x_high, seed=data_seed, device=DEVICE, dtype=DTYPE)

    rng = np.random.default_rng(int(data_seed + 123456)); idx = rng.permutation(int(n))
    n_train = int(round(float(train_frac) * int(n))); train_idx, test_idx = idx[:n_train], idx[n_train:]
    n_diag = max(1, int(round(float(diagnostic_frac) * n_train))); diag_idx = train_idx[:n_diag]
    ti = torch.as_tensor(train_idx, device=DEVICE); di = torch.as_tensor(diag_idx, device=DEVICE); te = torch.as_tensor(test_idx, device=DEVICE)

    print(f"\nExperiment\n  n/p/active : {n}/{p}/{n_active}\n  teacher    : {activation} | {structure} | {n_true_units} units"
          + (f" | {features_per_unit} features/unit" if structure == "mixed" else "")
          + f"\n  extra      : interaction={n_interactions} | quadratic={n_quadratic}"
          + f"\n  fitted BNN : {tuple(hidden_dims)} | {selection_mode}"
          + f"\n  flow       : K={K_flow} | {iaf_ordering_scheme} | slab_init={slab_init}"
          + f"\n  training   : epochs={epochs} | warmup={warmup_epochs} | R={R_train}/{R_diag}/{R_final}"
          + f"\n  seed       : data={data_seed} | fit={fit_seed}")

    out = train_trace(
        X[ti], y[ti], X[di], signal[di], X[te], signal[te], truth=info,
        selection_mode=selection_mode, hidden_dims=tuple(hidden_dims), sigma2=sigma2, init_sd=init_sd,
        K_flow=K_flow, flow_hidden_units=flow_hidden_units, flow_hidden_layers=flow_hidden_layers,
        scale_clip=scale_clip, flow_seed=int(fit_seed + 17), iaf_ordering_scheme=iaf_ordering_scheme,
        iaf_shuffle_within_role=iaf_shuffle_within_role, gate_type=gate_type, gate_scale=gate_scale,
        epochs=epochs, warmup_epochs=warmup_epochs, lr=lr, R_train=R_train, R_diag=R_diag, R_final=R_final,
        record_every=record_every, init_loc_jitter=init_loc_jitter, grad_clip=grad_clip,
        support_threshold=support_threshold, seed=fit_seed, slab_init=slab_init,
        slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd)
    out["data_info"] = info
    print("\nFinal result")
    display(pd.DataFrame.from_dict(out["result"], orient="index", columns=["value"]).round(6))
    return out

print("simfun, train_trace, run_experiment ready")


simfun, train_trace, run_experiment ready


## One controlled MLP run

In [4]:
exp = run_experiment(
    n=5000, p=300, n_active=10, n_true_units=4,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=1,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=3, epochs=1200, warmup_epochs=300, record_every=100,
    data_seed=400, slab_init="auto",
)

display(exp["history"].round(4))

NameError: name 'run_experiment' is not defined

In [8]:
# trig + 2 interactions
exp_trig_int = run_experiment(
    n=5000, p=200, n_active=10, n_true_units=6,
    activation="trig", structure="mixed", features_per_unit=2,
    n_interactions=2, n_quadratic=0,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=6, epochs=1200, warmup_epochs=300,
    R_train=32, R_diag=128, R_final=500,
    data_seed=400
)


Experiment
  n/p/active : 5000/200/10
  teacher    : trig | mixed | 6 units | 2 features/unit
  extra      : interaction=2 | quadratic=0
  fitted BNN : (20, 20) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100634
epoch=0000 init   reprR2=+0.0002 gatedR2=+0.0002 predVar=0.000796 U/V/tauSD=0.012/0.494/0.5 active=0.520
epoch=0001 repr   reprR2=+0.0018 gatedR2=+0.0018 predVar=0.000859 U/V/tauSD=0.0121/0.499/0.515 active=0.543
epoch=0100 repr   reprR2=+0.2845 gatedR2=+0.2845 predVar=0.317 U/V/tauSD=0.107/0.988/1.11 active=0.560
epoch=0200 repr   reprR2=+0.6057 gatedR2=+0.6057 predVar=0.4 U/V/tauSD=0.362/0.986/1 active=0.513
epoch=0300 repr   reprR2=+0.5165 gatedR2=+0.5165 predVar=0.143 U/V/tauSD=0.439/0.974/1.05 active=0.494
epoch=0400 select reprR2=-1.6039 gatedR2=+0.7408 predVar=34.5 U/V/tauSD=0.902/0.566/0.339 active=0.216
epoch=0500 select reprR2=-1.3521 gatedR2=+0.7213 pre

,value
mse,0.304712
r2,0.872570
train_time_sec,248.386345
tpr,0.900000
fpr,0.000000
accuracy,0.995000
auroc,0.940789
brier_bal,0.050002
expected_support,9.131999
selected_support,9.000000


In [7]:
# trig + 2 interactions
exp_trig_int = run_experiment(
    n=5000, p=200, n_active=10, n_true_units=6,
    activation="trig", structure="mixed", features_per_unit=2,
    n_interactions=2, n_quadratic=0,
    hidden_dims=(10, 10), selection_mode="feature_unit_induced_edge",
    K_flow=6, epochs=1200, warmup_epochs=300,
    R_train=32, R_diag=128, R_final=500,
    data_seed=400
)


Experiment
  n/p/active : 5000/200/10
  teacher    : trig | mixed | 6 units | 2 features/unit
  extra      : interaction=2 | quadratic=0
  fitted BNN : (10, 10) | feature_unit_induced_edge
  flow       : K=6 | cyclic3 | slab_init=auto
  training   : epochs=1200 | warmup=300 | R=32/128/500
  seed       : data=400 | fit=100634
epoch=0000 init   reprR2=-0.0000 gatedR2=-0.0000 predVar=0.000629 U/V/tauSD=0.0117/0.497/0.499 active=0.426
epoch=0001 repr   reprR2=+0.0006 gatedR2=+0.0006 predVar=0.000526 U/V/tauSD=0.0118/0.5/0.505 active=0.470
epoch=0100 repr   reprR2=+0.5102 gatedR2=+0.5102 predVar=0.223 U/V/tauSD=0.0683/0.993/1.17 active=0.367
epoch=0200 repr   reprR2=+0.5372 gatedR2=+0.5372 predVar=0.249 U/V/tauSD=0.193/0.959/0.955 active=0.429
epoch=0300 repr   reprR2=+0.5711 gatedR2=+0.5711 predVar=0.39 U/V/tauSD=0.26/0.982/0.923 active=0.481
epoch=0400 select reprR2=+0.2478 gatedR2=+0.7324 predVar=8.4 U/V/tauSD=0.641/0.411/0.384 active=0.274
epoch=0500 select reprR2=-1.6603 gatedR2=+0.75

,value
mse,0.329398
r2,0.862246
train_time_sec,196.784203
tpr,0.900000
fpr,0.000000
accuracy,0.995000
auroc,0.940000
brier_bal,0.050002
expected_support,9.132000
selected_support,9.000000
